# EvacOS-MA Training Pipeline

This notebook runs the default GRPO training path end-to-end on T4-class hardware.

**Sections:**
1. Setup
2. Config
3. Mount Drive (optional)
4. Smoke Rollout
5. Train
6. Eval
7. Export

## 1. Setup

In [14]:
import os

IN_COLAB = 'google.colab' in str(get_ipython()) if hasattr(__builtins__, 'get_ipython') else False

USE_UNSLOTH = True
USE_VLLM = True

if IN_COLAB:
    !pip uninstall -y transformers trl peft accelerate bitsandbytes vllm unsloth unsloth_zoo >/dev/null 2>&1

    !pip install -q \
        "torch" \
        "transformers>=4.56,<5.0" \
        "trl" \
        "peft" \
        "accelerate" \
        "bitsandbytes" \
        "datasets"

    !pip install -q \
        "pydantic>=2,<3" \
        "fastapi>=0.115" \
        "uvicorn>=0.30" \
        "numpy>=1.26" \
        "pyyaml" \
        "nbformat" \
        "wandb>=0.19"

    if USE_UNSLOTH:
        !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
        !pip install -q --no-deps "trl" "peft" "accelerate" "bitsandbytes"

    if USE_VLLM:
        !pip install -q "vllm"

else:
    print("Not in Colab — ensure training deps are installed manually.")

print(f"Setup complete. USE_UNSLOTH={USE_UNSLOTH}, USE_VLLM={USE_VLLM}")


/bin/bash: line 1: 3: No such file or directory
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.19.1 requires transformers!=5.0.*,!=5.1.*,!=5.2.*,!=5.3.*,!=5.4.*,!=5.5.0,>=4.56.0, but you have transformers 5.5.0 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.8 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.5.4 which is incompatible.
Setup complete. USE_UNSLOTH=True, USE_VLLM=True


In [4]:
import os
%cd /content

if not os.path.exists("/content/EvacOS2"):
    !git clone https://github.com/sai-shashankN/EvacOS2.git

%cd /content/EvacOS2


/content
Cloning into 'EvacOS2'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 151 (delta 10), reused 150 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 189.66 KiB | 9.03 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/EvacOS2


## 2. Config

In [16]:
from pathlib import Path
import json

from training.train import _load_yaml_config

base_config_path = Path('training/config.yaml')
config_dict = _load_yaml_config(base_config_path)

# --- User overrides (edit these) ---
MODEL_BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
EPISODES_PER_STEP = 4
CHECKPOINT_ROOT = '/content/drive/MyDrive/evacos2/checkpoints' if IN_COLAB else 'outputs/training/checkpoints'

config_dict['model']['base'] = MODEL_BASE
config_dict['rollout']['episodes_per_step'] = EPISODES_PER_STEP
config_dict['checkpoint']['root_dir'] = CHECKPOINT_ROOT

# Backend toggle. USE_UNSLOTH / USE_VLLM are set in the Setup cell.
# Unsloth + vLLM is the Colab-T4 fast path (5-10x faster than HF/TRL).
if USE_UNSLOTH:
    config_dict['backend'] = 'unsloth'
else:
    config_dict['backend'] = 'hf'
config_dict['rollout']['use_vllm'] = bool(USE_VLLM)

def dump_simple_yaml(value, indent=0):
    prefix = ' ' * indent
    if isinstance(value, dict):
        lines = []
        for key, item in value.items():
            if isinstance(item, (dict, list)):
                lines.append(f'{prefix}{key}:')
                lines.append(dump_simple_yaml(item, indent + 2))
            else:
                lines.append(f'{prefix}{key}: {json.dumps(item)}')
        return '\n'.join(lines)
    if isinstance(value, list):
        lines = []
        for item in value:
            if isinstance(item, (dict, list)):
                lines.append(f'{prefix}-')
                lines.append(dump_simple_yaml(item, indent + 2))
            else:
                lines.append(f'{prefix}- {json.dumps(item)}')
        return '\n'.join(lines)
    return f'{prefix}{json.dumps(value)}'

temp_config_dir = Path('/tmp' if IN_COLAB else 'outputs/notebook_runtime')
temp_config_dir.mkdir(parents=True, exist_ok=True)
config_path = temp_config_dir / 'training.notebook.override.yaml'
config_path.write_text(dump_simple_yaml(config_dict) + '\n', encoding='utf-8')

print(f'Model: {MODEL_BASE}')
print(f'Backend: {config_dict["backend"]}  |  use_vllm: {config_dict["rollout"]["use_vllm"]}')
print(f'Episodes per step: {EPISODES_PER_STEP}')
print(f'Checkpoint root: {CHECKPOINT_ROOT}')
print(f'Runtime config written to: {config_path}')

Model: Qwen/Qwen2.5-1.5B-Instruct
Backend: unsloth  |  use_vllm: True
Episodes per step: 4
Checkpoint root: /content/drive/MyDrive/evacos2/checkpoints
Runtime config written to: /tmp/training.notebook.override.yaml


# SECRETS

In [17]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["HF_ADAPTER_REPO"] = "sai-shashankN/evacos-2"
os.environ["WANDB_PROJECT"] = "EvacOS2"


## 3. Mount Drive (optional)

In [15]:
if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
        print('Drive mounted.')
    except Exception as e:
        print(f'Drive mount failed: {e}')
        print('Using local Colab storage at /content/outputs/')
else:
    print('Skipping Drive mount (not in Colab).')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## 4. Smoke Rollout

Validate the env end-to-end with StubPolicy before loading the model.

In [18]:
from evacos_ma.env import EvacEnvironment
from evacos_ma.models import DisasterType
from training.policy_adapter import StubPolicy
from training.rollout import collect_batch
from curriculum.controller import CurriculumController

env = EvacEnvironment()
policy = StubPolicy(seed=0)
curriculum = CurriculumController()

rng = __import__('random').Random(42)

results = collect_batch(
    env,
    policy,
    curriculum,
    num_episodes=2,
    seed_generator=lambda: rng.randint(0, 2_147_483_647),
    disaster_families=[DisasterType.fire],
    max_rounds=3,
)

for i, r in enumerate(results):
    print(f'Episode {i}: rounds={r.num_rounds}, samples={len(r.samples)}, '
          f'orch_raw_reward={r.total_raw_reward_by_role.get("orchestrator", 0):.4f}')

assert len(results) == 2, f'Expected 2 episodes, got {len(results)}'
print('Smoke rollout PASSED.')

Episode 0: rounds=3, samples=18, orch_raw_reward=0.0000
Episode 1: rounds=3, samples=18, orch_raw_reward=0.0000
Smoke rollout PASSED.


## 5. Train

**Unsloth + vLLM**: 5–10× faster than HF/TRL on Colab T4. 4-bit quant. Resume-safe with Phase 7 checkpoints (adapter format is PEFT-compatible across both backends).

In [ ]:
from training.train import run_training

# This will load the model, run GRPO training, checkpoint every 10 steps,
# and safe-early-stop on KeyboardInterrupt.
try:
    run_training(config_path=config_path)
except RuntimeError as e:
    if 'not installed' in str(e):
        print(f'Training skipped: {e}')
        print('Install requirements-training.txt and retry.')
    else:
        raise

## 6. Eval

In [ ]:
from training.policy_adapter import hf_policy_factory
from training.rollout import collect_batch
from training.reward import RewardNormalizer

runtime_config = _load_yaml_config(config_path)
eval_tiers = runtime_config['eval']['tiers']
eval_seeds = runtime_config['eval']['seeds']
eval_families = [DisasterType(name) for name in runtime_config['rollout']['disaster_families']]
latest_adapter = Path(runtime_config['checkpoint']['root_dir']) / 'latest' / 'lora_adapter'
policy = hf_policy_factory(runtime_config['model']['base'], lora_adapter_path=str(latest_adapter))
env = EvacEnvironment()
normalizer = RewardNormalizer()

class FixedTierCurriculum:
    def __init__(self, tier):
        self.tier = tier
    def suggest_next_tier(self, disaster_family):
        return self.tier
    def record_outcome(self, *args, **kwargs):
        return None

eval_results = []
for tier in eval_tiers:
    curriculum = FixedTierCurriculum(tier)
    ordered_seeds = []
    for seed in eval_seeds:
        ordered_seeds.extend([seed] * len(eval_families))
    seed_iter = iter(ordered_seeds)
    eval_results.extend(
        collect_batch(
            env,
            policy,
            curriculum,
            num_episodes=len(ordered_seeds),
            seed_generator=lambda: next(seed_iter),
            disaster_families=eval_families,
            max_rounds=10,
            is_eval=True,
            normalizer=normalizer,
        )
    )

print(f'Eval episodes: {len(eval_results)}')
for r in eval_results:
    print(f'  tier={r.tier} seed={r.seed} family={r.disaster_family} rounds={r.num_rounds} '
          f'orch_norm={r.total_normalized_reward_by_role.get("orchestrator", 0):.4f}')

# Plot per-role reward curve from metrics.csv if available
import csv
from pathlib import Path

metrics_csv = Path('outputs/training/metrics.csv')
if metrics_csv.exists():
    with open(metrics_csv) as f:
        reader = list(csv.DictReader(f))
    if reader:
        steps = [int(r['step']) for r in reader]
        orch_norm = [float(r['mean_norm_reward_orch']) for r in reader]
        floor_norm = [float(r['mean_norm_reward_floor']) for r in reader]
        print(f'\nMetrics rows: {len(reader)}')
        print(f'Latest step: {steps[-1] if steps else "N/A"}')
else:
    print('No metrics.csv found (training may not have run).')

## 7. Export

In [ ]:
import shutil

output_dir = Path('outputs')
export_path = '/content/evacos_ma_training_outputs.zip' if IN_COLAB else 'evacos_ma_training_outputs.zip'

if output_dir.exists():
    shutil.make_archive(export_path.replace('.zip', ''), 'zip', '.', 'outputs')
    print(f'Exported to {export_path}')
else:
    print('No outputs/ directory to export.')